# 第 05 章 · 执行器：进程、终端与资源生命周期

工具返回给模型时，子进程可能还在运行。进程退出时，输出管道可能还没排空；等待者取消时，
进程也不一定应该终止。执行器因此需要拥有独立于单次请求的 Session：进程对象、输出收集器、
输入句柄、截止时间和清理回执必须由同一个生命周期负责人管理。

本章实际创建受控 Python 子进程、管道和 PTY，比较 **等待结束、进程退出、输出结束、清理确认**。
需要 macOS/Linux 等 POSIX 环境及 `/bin/ps`；Windows 原生环境不支持这个原型。
这是普通本地执行器，**没有沙箱**，只执行下列已知探针。清理只覆盖受控进程组，不能约束主动 `setsid()` 逃逸的进程。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
import os
import time
import threading
if os.name != 'posix':
    raise RuntimeError('本章真实进程实验需要 POSIX；当前环境 blocked')
processes = load_example('05', 'process_probe')
run_dir = fresh_run_dir('ch05')
manager = processes.Manager(run_dir)
def wait_for(predicate, seconds=3.0):
    until = time.monotonic() + seconds
    while time.monotonic() < until:
        if predicate():
            return
        time.sleep(0.01)
    raise TimeoutError('探针条件未在限时内满足')

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. 返回一段输出，不等于进程已结束


    创建一个先输出、后等待输入的进程。第一次 poll 只读取当前可用数据，不拥有终止进程的权力。
    字节游标让后续读取从同一日志位置继续；输出预算只限制一次响应，不等于丢掉原始捕获。
    本教学原型开放普通 pipe stdin；Codex 模型工具对非 TTY 输入的限制不同，不能把底层能力当成产品开放能力。


In [3]:
first_budget = 5
handle = manager.start([sys.executable, '-u', '-c',
    "import sys; print('ready-for-input', flush=True); line=sys.stdin.readline(); print('got:'+line.strip(), flush=True)"], timeout=5)
session = manager.get(handle)
try:
    wait_for(lambda: session.stored >= len(b'ready-for-input\n'))
    first = session.poll(0, budget=first_budget)
    assert first['data'] == b'ready-for-input\n'[:first_budget]
    assert not first['root_exited'] and first['has_more']
    accepted = session.write(b'continue\n')
    assert accepted == len(b'continue\n')  # 内核接收，不是应用确认
    wait_for(lambda: session.fds_closed)
    tail = session.poll(first['next_cursor'], budget=2048)
    complete = first['data'] + tail['data']
    assert b'got:continue' in complete and tail['exit_code'] == 0
    assert tail['capture_complete']
finally:
    receipt = session.close()
assert receipt['confirmed_for_process_group'], receipt
{'first_chunk': first['data'], 'full_output': complete, 'cleanup': receipt}


{'first_chunk': b'ready',
 'full_output': b'ready-for-input\ngot:continue\n',
 'cleanup': {'root_exited': True,
  'exit_code': 0,
  'members': [],
  'fds_closed': True,
  'collector_joined': True,
  'confirmed_for_process_group': True,
  'scope': 'owned_process_group_only; setsid escape not contained',
  'reader_error': None}}

## 2. 响应截断与捕获丢失必须分开


    第一个进程输出全部进入日志，但每次只读少量字节；沿游标读完，能恢复完整数据。
    第二个进程触发总捕获上限，超出部分确实丢失。两者若都只写“truncated”，模型就不知道继续 poll 是否有用。
    调整 `payload_size` 和 `page_bytes`，比较分页次数与捕获计数。


In [4]:
payload_size, page_bytes = 4096, 137
normal = manager.get(manager.start([sys.executable, '-u', '-c', f"print('x'*{payload_size}, end='')"], timeout=3))
try:
    wait_for(lambda: normal.fds_closed)
    cursor, pieces = 0, []
    while True:
        page = normal.poll(cursor, budget=page_bytes)
        pieces.append(page['data'])
        cursor = page['next_cursor']
        if not page['has_more']:
            break
    assert b''.join(pieces) == b'x' * payload_size
    assert page['capture_lost_bytes'] == 0 and page['capture_complete']
finally:
    normal_receipt = normal.close()
limited = manager.get(manager.start([sys.executable, '-u', '-c', f"print('y'*{payload_size}, end='')"], timeout=3, spool_limit=64))
try:
    wait_for(lambda: limited.fds_closed)
    lost = limited.poll(0, budget=2048)
    assert len(lost['data']) == 64
    assert lost['capture_lost_bytes'] == payload_size - 64
    assert not lost['capture_complete']
finally:
    limited_receipt = limited.close()
assert normal_receipt['confirmed_for_process_group'] and limited_receipt['confirmed_for_process_group']
{'pages': len(pieces), 'response_only_truncation_lost_bytes': page['capture_lost_bytes'], 'capture_loss': lost['capture_lost_bytes']}


{'pages': 30, 'response_only_truncation_lost_bytes': 0, 'capture_loss': 4032}

## 3. 取消等待者，不等于取消进程


    同一 Session 可以被多次观察。某个等待者不再需要结果，应结束它自己的等待；由执行 owner 决定是否终止进程。
    这里先设置等待取消标志，核对子进程仍活着，然后显式 close，并检查回收回执。
    已关闭句柄拒绝写入，以免迟到输入进入别的进程。


In [5]:
sleeper = manager.get(manager.start([sys.executable, '-u', '-c', "import time; print('alive', flush=True); time.sleep(10)"], timeout=5))
try:
    wait_for(lambda: sleeper.stored >= len(b'alive\n'))
    cancel_wait = threading.Event()
    cancel_wait.set()
    cancelled_poll = sleeper.poll(sleeper.stored, wait=1, wait_cancel=cancel_wait)
    assert cancelled_poll['wait_cancelled']
    assert sleeper.process.poll() is None
finally:
    cleanup = sleeper.close()
assert cleanup['confirmed_for_process_group'], cleanup
closed = expect_error(ValueError, lambda: sleeper.write(b'late\n'), 'closed_process_handle')
{'waiting_cancelled': cancelled_poll['wait_cancelled'], 'cleanup': cleanup, 'late_write': closed}


{'waiting_cancelled': True,
 'cleanup': {'root_exited': True,
  'exit_code': -15,
  'members': [],
  'fds_closed': True,
  'collector_joined': True,
  'confirmed_for_process_group': True,
  'scope': 'owned_process_group_only; setsid escape not contained',
  'reader_error': None},
 'late_write': 'closed_process_handle'}

## 4. 总截止时间由 owner 持有


    如果每次 poll 都重新给进程五秒，总寿命便可无限延长。Session 在启动时固定 deadline，后台收集器负责观察它。
    本例不靠模型反复轮询来触发超时；随后读取结果，必须同时看到 timeout 与清理证据。
    `exit_code < 0` 是被信号终止的进程事实，不应改写成正常完成。


In [6]:
lifetime_seconds = 0.20
timed = manager.get(manager.start([sys.executable, '-u', '-c', 'import time; time.sleep(10)'], timeout=lifetime_seconds))
try:
    wait_for(lambda: timed.fds_closed, seconds=4)
    timed_result = timed.poll()
    assert timed_result['stop_reason'] == 'timeout'
    assert timed_result['exit_code'] is not None and timed_result['exit_code'] != 0
finally:
    timed_cleanup = timed.close()
assert timed_cleanup['confirmed_for_process_group'], timed_cleanup
{'stop_reason': timed_result['stop_reason'], 'exit_code': timed_result['exit_code'], 'cleanup_confirmed': timed_cleanup['confirmed_for_process_group']}


{'stop_reason': 'timeout', 'exit_code': -15, 'cleanup_confirmed': True}

## 5. PTY 是终端，不只是另一根管道


    PTY 会引入终端回显、行规程、窗口尺寸等行为。应用是否认为自己连接着终端，可以通过 `isatty` 直接观察。
    本例实际修改窗口尺寸、输入一行并读取响应；不要断言输出严格等于输入，因为 PTY 可能回显并把换行转换成 CRLF。


In [7]:
terminal = manager.get(manager.start([sys.executable, '-u', '-c',
    "import os,sys; print('tty='+str(sys.stdin.isatty()),flush=True); x=input(); print('answer='+x,flush=True)"], tty=True, timeout=5))
try:
    terminal.resize(30, 100)
    wait_for(lambda: b'tty=True' in terminal.poll(0, budget=2048)['data'])
    terminal.write(b'hello\n')
    wait_for(lambda: terminal.fds_closed)
    terminal_result = terminal.poll(0, budget=2048)
    assert b'answer=hello' in terminal_result['data']
    assert terminal_result['exit_code'] == 0
finally:
    terminal_cleanup = terminal.close()
assert terminal_cleanup['confirmed_for_process_group'], terminal_cleanup
all_receipts = manager.close_all()
assert all(r['confirmed_for_process_group'] for r in all_receipts.values()), all_receipts
{'pty_output': terminal_result['data'], 'closed_sessions': len(all_receipts), 'logs': str(run_dir)}


{'pty_output': b'tty=True\r\nhello\r\nanswer=hello\r\n',
 'closed_sessions': 6,
 'logs': '<project>/runs/notebooks/ch05-20261113T124115Z-1d67aa14'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [exec_command_inner](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/unified_exec/process_manager.rs#L603-L672)：初始exec在收集输出之前保存仍活跃进程，以及输出尚未关闭的远端退出进程；yield时间与one-shot completion timeout分别选择。

- [write_stdin_inner / prepare_process_handles](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/unified_exec/process_manager.rs#L904-L1025)：stdin/poll按原进程对象的interaction_lock串行；审批前及审批后校验同一个Arc对象，避免排队请求写入复用ID的新进程。

- [OutputBuffers / push_chunk](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/unified_exec/process.rs#L43-L68)：pending和transcript在同一锁内追加；二者都是有上限的HeadTailBuffer。


## 现在应能解释什么


    能解释为什么一次 exec 响应不能代替完整进程状态，为什么输出游标、采集上限和总期限要分开，
    为什么取消观察者与清理执行资源属于不同操作。

    例子复用作者的 POSIX 局部原型。它保存完整捕获范围内的字节日志；Codex 的缓冲策略不同。
    本章没有验证敌意后代进程、跨平台执行或 OS 隔离；观察到的清理证据仅覆盖这些受控进程组。
